In [17]:
import winsound
from tqdm.auto import tqdm
import sympy as sp
from sympy.core.function import AppliedUndef
from sympy.physics.wigner import clebsch_gordan
from sympy.physics.quantum.spin import Rotation as rot
from sympy import pi
from random import randint
import psutil
import dill
import os
import itertools

sp.init_printing(use_unicode=True, wrap_line=True, num_columns=120)

import time
start_time = time.time()

Jmax = 3
Lmax = 2

# --- Definizione simboli ---
J, LB, LR, L1, L2, JX, lam, lam1, lam2, eta = sp.symbols('J LB LR L1 L2 JX lam lam1 lam2 eta', integer=True)
theta1, theta2, thetal, thetaK, phi, chi = sp.symbols('theta1 theta2 thetal thetaK phi chi', real=True)
eta_par = sp.symbols('eta_par', integer=True)
Res = sp.symbols('Res')

# --- Helicity amplitudes e altri coefficienti simbolici ---
H = sp.Function('H', complex=True)(eta, J, lam, LB)     # Helicity amplitude per B -> K_J
Hbar = sp.Function('Hbar', complex=True)(eta, J, lam, LB)     # Helicity amplitude per B -> K*_J
LHA = sp.Function('LHA', real=True)(eta_par, lam1, lam2)  # Leptonic Helicity amplitude
G = sp.Function('G', complex=True)(eta, J, lam, LB)  # Helicity Amplitude for K_J -> R P1

# --- Breit-Wigner propagatore ---
Prop = sp.Function('R')(Res)

# --- Blatt-Weisskopf factor ---
L, rBW = sp.symbols('L rBW', positive=True, real=True)
rB, r3, r2 = sp.symbols('rB r3 r2', positive=True, real=True)
q = sp.symbols('q', real=True)
qR = sp.symbols('qR', real=True)

# funzione simbolica generica
BWfunc = sp.Function('BW', real=True)

# istanza base (con argomenti simbolici)
BW = BWfunc(L, rBW, qR)

atot = []
a_NR = sp.symbols('a_NR', complex=True)
atot.append(a_NR)  # contributo non-risonante

In [18]:
# --- Costruzione dell'ampiezza AR ---
AR = 0
J_val = 0

lambda1 = sp.symbols('lambda1', real=True)
lambda2 = sp.symbols('lambda2', real=True)
lam_val = 0
    
Hterm = H.subs({J: J_val, lam: lam_val, LB: 1})
term_vector = (
    -Hterm * LHA.subs(eta_par, eta).subs(lam1, lambda1).subs(lam2, lambda2) *
    rot.d(1, lam_val, lambda1 - lambda2, thetal)
)
AR += term_vector

term_timelike = 0
t = sp.symbols('t', real=True)
Ht = sp.symbols('Ht', complex=True)
Lt = sp.Function('Lt', complex=True)(lam1, lam2)     # Helicity amplitude per B -> K_J
term_timelike = (
    Ht * Lt.subs(lam1, lambda1).subs(lam2, lambda2)
)

AR += term_timelike


# Mostra l'ampiezza totale
print("\n" + "="*50)
if AR == 0:
    print("L'ampiezza AR è nulla (nessuna risonanza selezionata).")
else:
    print("Ampiezza AR totale:")
    print(f"Termini nell'ampiezza AR: {len(AR.as_ordered_terms())}")
print("="*50)

# Mostra l'ampiezza
sp.pprint(AR)


Ampiezza AR totale:
Termini nell'ampiezza AR: 2
                                              1                
Ht⋅Lt(λ₁, λ₂) - H(η, 0, 0, 1)⋅LHA(η, λ₁, λ₂)⋅d         (thetal)
                                              0,λ₁ - λ₂        


In [19]:
b = sp.symbols('b', real = True)
ml = sp.symbols('ml', real = True)

Leptonic_subs = {
    ## Leptonic Helicity Amplitudes (Vettoriali - Standard)
    LHA.subs({eta_par: -1, lam1: sp.Rational(-1,2), lam2: sp.Rational(1,2)}): (1+b)*sp.sqrt(2)*q,
    LHA.subs({eta_par: -1, lam1: sp.Rational(1,2), lam2: sp.Rational(-1,2)}): (1-b)*sp.sqrt(2)*q,
    LHA.subs({eta_par: -1, lam1: sp.Rational(-1,2), lam2: sp.Rational(-1,2)}): 2*ml,
    LHA.subs({eta_par: -1, lam1: sp.Rational(1,2), lam2: sp.Rational(1,2)}): 2*ml,

    LHA.subs({eta_par: 1, lam1: sp.Rational(-1,2), lam2: sp.Rational(1,2)}): (1-b)*sp.sqrt(2)*q,
    LHA.subs({eta_par: 1, lam1: sp.Rational(1,2), lam2: sp.Rational(-1,2)}): (1+b)*sp.sqrt(2)*q,
    LHA.subs({eta_par: 1, lam1: sp.Rational(-1,2), lam2: sp.Rational(-1,2)}): 2*ml,
    LHA.subs({eta_par: 1, lam1: sp.Rational(1,2), lam2: sp.Rational(1,2)}): 2*ml,

    Lt.subs({lam1: sp.Rational(-1,2), lam2: sp.Rational(-1,2)}): 2*ml,
    Lt.subs({lam1: sp.Rational(1,2), lam2: sp.Rational(1,2)}): -2*ml,
    Lt.subs({lam1: sp.Rational(-1,2), lam2: sp.Rational(1,2)}): 0,
    Lt.subs({lam1: sp.Rational(1,2), lam2: sp.Rational(-1,2)}): 0,
}
# AMPIEZZA TOTALE
Atot = AR
Atotsqfin = 0

for lambda1_val in [sp.Rational(-1,2), sp.Rational(1,2)]:
    for lambda2_val in [sp.Rational(-1,2), sp.Rational(1,2)]:
        
        Term_L = Atot.subs({eta: -1, lambda1: lambda1_val, lambda2: lambda2_val})
        Term_R = Atot.subs({eta: 1, lambda1: lambda1_val, lambda2: lambda2_val})
        
        Amp_Total_Helicity = Term_L + Term_R
        Amp_Total_Helicity = Amp_Total_Helicity.doit().subs(Leptonic_subs)
        
        term_sq = Amp_Total_Helicity * sp.conjugate(Amp_Total_Helicity)
        
        Atotsqfin += term_sq
N = (8)
Atotsqfin = sp.expand((Atotsqfin)/N)

terms_list = Atotsqfin.as_ordered_terms()
print(f"Numero di termini: {len(terms_list)}\n")
print(f"Numero di parametri a_i: {len(atot)}\n")

sp.pprint(Atotsqfin)

Numero di termini: 13

Numero di parametri a_i: 1

               2  2                   2         ______________    2  2                   2         _____________    2  ↪
       2 __   b ⋅q ⋅H(-1, 0, 0, 1)⋅sin (thetal)⋅H(-1, 0, 0, 1)   b ⋅q ⋅H(-1, 0, 0, 1)⋅sin (thetal)⋅H(1, 0, 0, 1)   b ⋅ ↪
4⋅Ht⋅ml ⋅Ht + ──────────────────────────────────────────────── - ─────────────────────────────────────────────── - ─── ↪
                                     4                                                  4                              ↪

↪  2                  2         ______________    2  2                  2         _____________                        ↪
↪ q ⋅H(1, 0, 0, 1)⋅sin (thetal)⋅H(-1, 0, 0, 1)   b ⋅q ⋅H(1, 0, 0, 1)⋅sin (thetal)⋅H(1, 0, 0, 1)     2                  ↪
↪ ──────────────────────────────────────────── + ────────────────────────────────────────────── + ml ⋅H(-1, 0, 0, 1)⋅c ↪
↪                     4                                                4                             

In [20]:
mB, mK = sp.symbols('mB mK', real=True, positive=True)
A = sp.Function('A', complex=True)(eta, lam)
FV,FA,FP = sp.symbols('FV FA FP', complex=True)
lam_kin = sp.Symbol("lambda", real=True, positive=True)
norm_long = sp.sqrt(lam_kin) / sp.sqrt(q**2)
scalar_kin_factor = (mB**2 - mK**2 + q**2) / sp.sqrt(q**2)
half = sp.Rational(1,2)

helicity_to_transverity_subs = {
    H.subs({eta: -1, J: 0, lam: 0, LB: 1}): half * norm_long*(FV - FA),
    H.subs({eta: 1, J: 0, lam: 0, LB: 1}): half * norm_long*(FV + FA),
    # Ht: (sp.sqrt(q**2) / (2*ml))*FP + norm_long * FA,
    Ht: half*((sp.sqrt(q**2)/(ml)) * FP + scalar_kin_factor * FA),
    sp.sin(thetal)**2: 1-sp.cos(thetal)**2,
}
Atotsq_mod = Atotsqfin.subs(helicity_to_transverity_subs).expand()

In [21]:
# Atotsq_mod = Atotsqfin

def extract_trig_structure(expr):
    """Estrae la struttura trigonometrica completa da un'espressione"""
    trig_factors = []
    non_trig_factors = []
    
    # Scompone il termine in fattori
    if expr.is_Mul: #controlla se è un prodotto 'Mul' per Sympy e li ordina come termini
        factors = expr.as_ordered_factors()
    elif expr.is_Pow:
        # Gestisci le potenze
        base, exp = expr.as_base_exp() # Divide in base ed esponente
        if base.atoms(sp.sin, sp.cos):
            return [expr], []  # Tutta la potenza è trigonometrica
        else:
            return [], [expr]  # Tutta la potenza è non-trigonometrica
    else:
        factors = [expr] #in questo caso ci sono più termini di tipo 'Add' sommati tra loro e li mette in una lista
    
    for factor in factors:
        if factor.atoms(sp.sin, sp.cos):
            # Questo fattore contiene funzioni trigonometriche
            if factor.is_Pow:
                # È una potenza di funzione trigonometrica
                trig_factors.append(factor)
            elif factor.is_Mul:
                # È un prodotto che contiene trigonometria
                sub_trig, sub_non_trig = extract_trig_structure(factor)
                trig_factors.extend(sub_trig) #extend aggiunge in coda tutti gli elementi della lista 'sub_trig'
                non_trig_factors.extend(sub_non_trig)
            else:
                # È una singola funzione trigonometrica
                trig_factors.append(factor)
        else:
            non_trig_factors.append(factor)
    
    return trig_factors, non_trig_factors

# Raccoglie tutti i prodotti trigonometrici unici
trig_coefficients = {}

print("Analizzando ogni termine dell'espressione...")
for i, term in tqdm(enumerate(Atotsq_mod.as_ordered_terms()), total=len(Atotsq_mod.as_ordered_terms()), desc="Analisi termini angolari"):
    
    # Estrai la struttura trigonometrica
    trig_factors, non_trig_factors = extract_trig_structure(term)
    
    # Costruisci il prodotto trigonometrico
    if trig_factors:
        trig_product = 1
        for tf in trig_factors:
            trig_product *= tf
        trig_key = str(trig_product)
    else:
        trig_key = "COSTANTE"
    
    # Costruisci il coefficiente non trigonometrico
    if non_trig_factors:
        non_trig_product = 1
        for ntf in non_trig_factors:
            non_trig_product *= ntf
    else:
        non_trig_product = 1
    
    # Accumula i coefficienti
    if trig_key in trig_coefficients:
        trig_coefficients[trig_key] += non_trig_product
    else:
        trig_coefficients[trig_key] = non_trig_product

print(f"\nTrovati {len(trig_coefficients)} prodotti trigonometrici distinti")

# print("Semplificando i coefficienti...")
# for key in tqdm(trig_coefficients, desc="Semplificazione coefficienti", total=len(trig_coefficients)):
#     trig_coefficients[key] = sp.simplify(trig_coefficients[key])


# Ordina per lunghezza della stringa (prima i più semplici)
print("Ordinando i prodotti trigonometrici per complessità")
sorted_keys = sorted(trig_coefficients.keys(), key=lambda x: (len(x), x))

print(f"\n" + "="*70)
print("RIEPILOGO FINALE:")
print("="*70)
print(f"- Prodotti trigonometrici distinti: {len([k for k, v in trig_coefficients.items() if v != 0])}")
for k in range(len(sorted_keys)):
    print(f"  • Prodotto #{k}: {sorted_keys[k]}")

print(f"- Analisi completata con successo!")


Analizzando ogni termine dell'espressione...


Analisi termini angolari:   0%|          | 0/18 [00:00<?, ?it/s]


Trovati 2 prodotti trigonometrici distinti
Ordinando i prodotti trigonometrici per complessità

RIEPILOGO FINALE:
- Prodotti trigonometrici distinti: 2
  • Prodotto #0: COSTANTE
  • Prodotto #1: cos(thetal)**2
- Analisi completata con successo!


In [22]:
for key in sorted_keys:
    coeff = trig_coefficients[key]
    for x in [FV,FA,FP]:
        coeff = coeff.subs(x*sp.conjugate(x), sp.Abs(x)**2)
        coeff = coeff.subs(sp.conjugate(x)*x, sp.Abs(x)**2)
    trig_coefficients[key] = coeff

In [23]:
for i, key in enumerate(sorted_keys):
    f = trig_coefficients[key]
    print(f"\n{'-'*50}\nProdotto #{i}: {key}\nCoefficiente:\n")
    sp.pprint(f.collect(sp.Abs(FV)).collect(sp.Abs(FA)).collect(sp.Abs(FP)).collect(sp.Abs(FP)**2).collect(FA*sp.conjugate(FP)).collect(FP*sp.conjugate(FA)))


--------------------------------------------------
Prodotto #0: COSTANTE
Coefficiente:

                                                                          2              ⎛ 2       4   2       2   2   ↪
   ⎛  2        2          2⎞ __      ⎛  2        2          2⎞ __   λ⋅│FV│     2     2   ⎜b ⋅λ   mB ⋅ml    2⋅mB ⋅mK ⋅m ↪
FA⋅⎝mB ⋅ml - mK ⋅ml + ml⋅q ⎠⋅FP + FP⋅⎝mB ⋅ml - mK ⋅ml + ml⋅q ⎠⋅FA + ─────── + q ⋅│FP│  + ⎜──── + ─────── - ─────────── ↪
                                                                       4                 ⎜ 4        2            2     ↪
                                                                                         ⎝         q            q      ↪

↪  2                 4   2                     ⎞      
↪ l        2   2   mK ⋅ml        2   2     2  2⎟     2
↪ ── + 2⋅mB ⋅ml  + ─────── - 2⋅mK ⋅ml  + ml ⋅q ⎟⋅│FA│ 
↪                     2                        ⎟      
↪                    q                         ⎠      

-----------------------------